# Presto: learn one field, then generalize

This notebook accompanies `experiments/presto_debug/train.py`. It starts with two deliberately tiny training-set checks, then compares an unregularized run and three regularization settings on a larger training/validation split. Every run writes TensorBoard events, a history table, predictions, a configuration and a checkpoint.

**Scope:** the existing inputs cover the full season. These are retrospective experiments, not pre-harvest forecasts. A model that memorizes one or two training fields has passed a software sanity check; it has not demonstrated accuracy on new fields.


## What enters the model

The DuckDB monthly observations contain Sentinel-2 reflectance bands and NDVI, aligned in calendar order. Missing months and unavailable sensor channels are masked. The pretrained Presto encoder maps a field's time series to an embedding; a new head with batch normalization turns that embedding into a yield estimate. The target is crop-cut yield in t/ha; MSE has units of (t/ha)². Preprocessing and any target statistics must use training data only.

The underlying Presto encoder retains its own architecture. Batch normalization is applied in the new prediction head; it does not replace normalization layers inside the pretrained encoder.

For stages with at least three training fields, the target is standardized using only the training subset mean and standard deviation. Predictions are transformed back to t/ha before computing or plotting the saved MSE and predictions; validation labels never determine the scaling. One- and two-field diagnostics use unscaled yield.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

# Works when started in the repo root, this notebook's folder, or a subfolder.
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / 'experiments' / 'presto_debug' / 'train.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open the notebook from the repository or one of its subdirectories.')
print('Repository:', ROOT)
print('Python:', sys.executable)
for package in ('torch', 'einops', 'duckdb', 'numpy', 'pandas', 'matplotlib', 'tensorboard'):
    print(f'{package:12}', 'available' if importlib.util.find_spec(package) else 'MISSING')
print('Trainer:', ROOT / 'experiments' / 'presto_debug' / 'train.py')

# A TensorBoard module can be present while viewer dependencies are missing.
try:
    import importlib
    importlib.import_module('tensorboard.main')
    print('TensorBoard viewer: ready')
except ImportError as exc:
    print('TensorBoard viewer: unavailable:', exc)
    print('Install complete dependencies with:', subprocess.list2cmdline([sys.executable, '-m', 'pip', 'install', '--upgrade', 'tensorboard']))


Use a kernel with PyTorch, einops, DuckDB, pandas, matplotlib and TensorBoard installed. The preflight imports the TensorBoard viewer as well as checking package presence: an environment may write event files while still missing viewer dependencies. If the viewer check fails, run the printed pip command in that same environment. The project-specific v2 database and verified pretrained checkpoint must also be available; the trainer checks the inputs and stops if they are not.


## Step 1: deliberately overfit one field

The first run uses one field from the training split, with no regularization. Its goal is to make the training loss very small, not to measure generalization. Standard batch normalization cannot estimate meaningful batch variance from one example; this run keeps its batch-normalization layer in evaluation mode and uses its fixed running statistics. This exception is recorded in the run configuration. The one-field check therefore tests the rest of the pipeline while retaining the same head architecture.

The target scaling for this sanity check must not center the single label to a trivial zero-error target. Check the recorded scaling and predictions in physical units.


## Step 2: overfit two fields

The second run uses two training fields. Batch normalization can now compute batch statistics during training. With only two fields its estimates remain unstable, and its training-mode predictions need not match evaluation-mode predictions immediately: inspect both the loss and the saved predictions. The single-field and two-field checks are diagnostics, not a held-out evaluation.


## Steps 3–6: establish a baseline, then increase regularization

After the tiny checks, run `none`, `light`, `medium`, and `strong`. The `none` stage uses the same 128 training fields and grouped validation fields as the regularized stages, with zero dropout and zero weight decay. This first shows whether the model can fit a realistic-sized training subset before regularization is increased. The trainer records the exact settings for every stage. Compare them on the **same grouped validation split** and seed.

An earlier exploratory set of 25-epoch regularization runs showed underfitting. The follow-up adds the unregularized baseline, uses training-only target standardization and extends the training budget. This is a revised exploratory protocol, so do not present its results as a preregistered confirmation. A growing train–validation gap can suggest overfitting; both losses remaining high may indicate underfitting, optimization difficulty or noisy labels. Validation is used for development and is not a final test.

A comparison with the one- or two-field runs changes the training set as well as regularization, so it cannot isolate the effect of regularization.


In [ ]:
from datetime import datetime
import subprocess

# Execution is opt-in: change RUN_TRAINING only after reviewing the cells and CLI.
RUN_TRAINING = False
STAGES = ['one', 'two']  # after sanity checks: ['none', 'light', 'medium', 'strong']
EPOCHS = None            # None uses each stage's configured default
SEED = 20261108

# To inspect an existing run, set RUN_ROOT to the directory containing stage folders.
RUNS_DIR = ROOT / 'experiments' / 'presto_debug' / 'runs'
existing = sorted((p for p in RUNS_DIR.iterdir() if p.is_dir() and
                   any((p / stage / 'history.csv').is_file()
                       for stage in ('one', 'two', 'none', 'light', 'medium', 'strong'))),
                  key=lambda p: p.stat().st_mtime) if RUNS_DIR.exists() else []
# Shipped results live under runs/latest. New training always gets a fresh directory.
if not RUN_TRAINING and (RUNS_DIR / 'latest').is_dir():
    RUN_ROOT = RUNS_DIR / 'latest'
elif not RUN_TRAINING and existing:
    RUN_ROOT = existing[-1]
else:
    RUN_ROOT = RUNS_DIR / datetime.now().strftime('%Y%m%d_%H%M%S')
print('Run root:', RUN_ROOT)
print('Training enabled:', RUN_TRAINING)

if RUN_TRAINING:
    for stage in STAGES:
        output = RUN_ROOT / stage
        if output.exists():
            raise FileExistsError(f'Refusing to overwrite {output}')
        command = [sys.executable, str(ROOT / 'experiments' / 'presto_debug' / 'train.py'),
                   '--stage', stage, '--output', str(output), '--seed', str(SEED)]
        if EPOCHS is not None:
            command += ['--epochs', str(EPOCHS)]
        print('Running:', subprocess.list2cmdline(command), flush=True)
        subprocess.run(command, cwd=ROOT, check=True)
else:
    print('No training started. Set RUN_TRAINING=True to run the selected stages.')


The output directories must be new. For a complete progression you can set `STAGES = ['one', 'two', 'none', 'light', 'medium', 'strong']`, or run each stage separately; use the same `RUN_ROOT` when inspecting them together. Set an explicit epoch limit only when you intend a shorter smoke test. A run stopped before convergence is not evidence that the network cannot memorize the examples.

When training is disabled, the notebook automatically selects the shipped `experiments/presto_debug/runs/latest/` results when present; otherwise it picks the most recently modified run. New training always uses a fresh timestamped directory. You can always set `RUN_ROOT` explicitly.


## Inspect the recorded settings and learning curves

Point `RUN_ROOT` at a completed run if you opened this notebook later. Each curve comes from the actual saved `history.csv`. Validation metrics are shown only when available; R² is undefined for fewer than two samples or a constant target.

`train_mse` uses evaluation mode; `fit_mode_mse` uses training-mode behavior for the tiny overfit tests. When BatchNorm is active these can differ. The gradient plots display logged encoder and head norms, not the parameter values.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

stage_order = {name: i for i, name in enumerate(('one', 'two', 'none', 'light', 'medium', 'strong'))}
stage_dirs = (sorted((p for p in RUN_ROOT.iterdir() if p.is_dir()),
                     key=lambda p: (stage_order.get(p.name, len(stage_order)), p.name))
              if RUN_ROOT.exists() else [])
if not stage_dirs:
    print('No stage directories found. Set RUN_ROOT to a completed run.')
for stage_dir in stage_dirs:
    history_file = stage_dir / 'history.csv'
    config_file = stage_dir / 'config.json'
    if not history_file.exists():
        print(f'{stage_dir.name}: history.csv not found')
        continue
    print(f'\nStage: {stage_dir.name}')
    if config_file.exists():
        display(json.loads(config_file.read_text(encoding='utf-8')))
    history = pd.read_csv(history_file)
    display(history.tail())
    fig, ax = plt.subplots(figsize=(8, 4))
    for column in ('train_mse', 'fit_mode_mse', 'val_mse'):
        if column in history and history[column].notna().any():
            ax.plot(history['epoch'], history[column], label=column)
    ax.set(title=f'{stage_dir.name}: loss', xlabel='Epoch', ylabel='MSE (t/ha)²')
    ax.legend()
    ax.grid(alpha=0.25)
    plt.show()

    gradient_columns = [c for c in ('encoder_grad_norm', 'head_grad_norm')
                        if c in history and history[c].notna().any()]
    if gradient_columns:
        fig, ax = plt.subplots(figsize=(8, 3))
        for column in gradient_columns:
            ax.plot(history['epoch'], history[column], label=column)
        ax.set(title=f'{stage_dir.name}: gradient norms', xlabel='Epoch', ylabel='L2 gradient norm')
        ax.legend()
        ax.grid(alpha=0.25)
        plt.show()


For the tiny checks, inspect final errors directly. With two points, training-mode batch statistics and evaluation-mode running statistics differ; check the saved stage settings and evaluation predictions rather than assuming that the final training loss is the evaluation error.


In [ ]:
for stage_dir in stage_dirs:
    prediction_file = stage_dir / 'predictions.csv'
    if not prediction_file.exists():
        print(f'{stage_dir.name}: predictions.csv not found')
        continue
    predictions = pd.read_csv(prediction_file)
    print(f'\nStage: {stage_dir.name}')
    display(predictions.head(10))
    fig, ax = plt.subplots(figsize=(5, 5))
    for split, rows in predictions.groupby('split'):
        ax.scatter(rows['observed'], rows['predicted'], s=20, alpha=0.65, label=str(split))
    values = pd.concat([predictions['observed'], predictions['predicted']]).dropna()
    if not values.empty:
        lo, hi = values.min(), values.max()
        ax.plot([lo, hi], [lo, hi], '--', color='gray')
    ax.set(xlabel='Observed yield (t/ha)', ylabel='Predicted yield (t/ha)', title=stage_dir.name)
    ax.legend()
    ax.grid(alpha=0.25)
    plt.show()


## TensorBoard

Each stage writes event files beneath `<stage>/tensorboard`. From a terminal in the same Python environment, run the command printed below and open the address TensorBoard reports. The log root is the parent of the stage directories, so TensorBoard can show the stages together. The saved CSV plots above are available without starting the server.

If the viewer import check above reports a missing dependency, run the printed `python -m pip install --upgrade tensorboard` command in the notebook kernel environment before launching. Merely having `torch.utils.tensorboard` installed is not sufficient to guarantee the viewer can start.


In [ ]:
import sys
import subprocess
command = [sys.executable, '-m', 'tensorboard.main', '--logdir', str(RUN_ROOT)]
print('In a terminal, run:')
print(subprocess.list2cmdline(command))
print('Then open the URL printed by TensorBoard, usually http://localhost:6006/')


## Reading the outcome

- If one field cannot be memorized, check input masks, target units, gradients, optimizer settings and whether the batch-normalization mode is correct before tuning regularization.
- If one field works but two do not, inspect predictions in both training and evaluation mode and the batch-normalization running statistics; increasing the batch size changes their stability.
- If training improves while grouped validation does not, compare the unregularized baseline with the three recorded regularization settings and report uncertainty before claiming an improvement.
- For a genuine pre-harvest forecast, rebuild all inputs as of a fixed issue date and use only labels available before that date. The historical full-season scores in this notebook do not answer that question.
